# NCUReport2JSON Conversion

In this notebook you will learn how to:

* Traverse the NCU Report
* Create a dict out of the traversed profile results (actions) and their metrics
* Save the created dict as a JSON file

## Setup

First, import NVIDIA Nsight Compute's Python Report Interface (PRI) as `ncu_report`
and load an `ncu-rep` report file with `load_report`.


In [ ]:
import ncu_report

report_file_path = "../sample_reports/mergeSort.ncu-rep"
report = ncu_report.load_report(report_file_path)

Second, define a helper function, `traverse_report`, that allows us to iterate over the report's structure (ranges, actions, metrics) using an iterator

In [ ]:
def traverse_report(report):
    for range_idx, my_range  in enumerate(report):
        for action_idx, action in enumerate(my_range):
            action_uniq_name = f"{action_idx}:{action}"
            for metric_name in action:
                yield (
                    (range_idx, action_uniq_name, metric_name),
                    action[metric_name],
                )

## Constructing a dictionary of the report

To convert the ncu into a json file, we first need to create a dictionary resembling the structure of the report.

In [ ]:
from collections import defaultdict

def report_to_dict(report):
    
    report_dict = defaultdict(lambda : defaultdict(lambda: defaultdict(dict)))

    for (range_idx, action_idx, metric_idx), metric in traverse_report(report):
        metric_dict = report_dict[range_idx][action_idx][metric_idx]
    
        if metric.has_value():
            metric_dict["value"] = metric.value()
    
        num_instances = metric.num_instances()
        if num_instances == 0:
            continue
        
        corr_ids = metric.correlation_ids()
        metric_dict["instances"] = {
            corr_ids.value(i) : metric.value(i) for i in range(num_instances)
        }

    return report_dict


report_dict = report_to_dict(report)

## Saving the generated json

Now that we have a copy of the report in a dictionary, we can save it as a json file

In [ ]:
import json

with open("mergeSort.json", "wt") as f:
    json.dump(report_dict, f)

> Note: A dictionary can serve as a foundation for conversion to many other formats. Therefore, the `report_to_dict` function can be reused to convert to formats that accept dictionaries, such as `YAML` (shown below).

In [ ]:
import yaml

with open("mergeSort.yaml", "wt") as f:
    yaml.dump(report_dict, f)